# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [1]:
GROUP_ID = 4
NOTEBOOK_URL = "https://github.com/matiuszaa/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Mateusz Baran / 304483",
    "Mikołaj Garbowski",
    "Jakub Lemański",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [2]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2 ** 30, 2))


Python: 3.12.10
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 20
RAM GiB: 31.77


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [4]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags",
        "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates",
         "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Application logs',
 'feature': 'status codes',
 'stress': 'selective filters and string columns'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [5]:
SCALE = "large"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 4 {'name': 'Application logs', 'feature': 'status codes', 'stress': 'selective filters and string columns'}
Rows: 50000000
Run seed recorded in manifest: 289088503403094465115627696573718415941
Output directory: ..\data\phase2_26L\group_04


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [6]:
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    n = df.height

    endpoints = ["/api/users", "/api/products", "/api/cart", "/api/checkout", "/api/search", "/api/auth"]
    methods = ["GET", "POST", "PUT", "DELETE"]

    status_codes = rng.choice(
        [200, 201, 400, 401, 403, 404, 500, 502, 503],
        size=n,
        p=[0.65, 0.05, 0.05, 0.05, 0.02, 0.15, 0.01, 0.01, 0.01]
    )

    level_map = {"A": "INFO", "B": "INFO", "C": "DEBUG", "D": "WARN", "E": "ERROR", "F": "FATAL"}

    df_custom = df.rename({
        "entity_id": "session_id",
        "category": "log_level",
    }).with_columns([
        pl.Series("status_code", status_codes, dtype=pl.Int32),
        pl.Series("endpoint", rng.choice(endpoints, size=n)),
        pl.Series("http_method", rng.choice(methods, size=n, p=[0.8, 0.15, 0.03, 0.02])),

        (pl.col("metric_1") * 25).cast(pl.Float32).alias("response_time_ms"),
        (pl.col("metric_2") * 10).cast(pl.Int32).alias("payload_size_bytes"),

        pl.col("log_level").replace(level_map),
        pl.when(pl.col("event_id") % 20 == 0).then(None).otherwise(pl.col("device")).alias("device")
    ]).drop(["metric_1", "metric_2"])

    return df_custom


def generate_dimension_table(card, rng):
    endpoints = ["/api/users", "/api/products", "/api/cart", "/api/checkout", "/api/search", "/api/auth"]

    return pl.DataFrame({
        "endpoint": endpoints,
        "microservice": ["user-service", "catalog-service", "order-service", "payment-service", "search-service",
                         "auth-service"],
        "team_owner": ["team-alpha", "team-beta", "team-gamma", "team-gamma", "team-beta", "team-alpha"],
        "sla_timeout_ms": [200, 300, 150, 500, 400, 100]
    })

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Generating events...")
base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)

print("Generating dimensions...")
dimension = generate_dimension_table(CARD, rng)

print("Saving parquet files...")
events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

print("Saving optimized parquet files...")
events.sort(["status_code", "endpoint"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2 ** 30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n--- Manifest ---")
print(json.dumps(manifest, indent=2))

Generating events...
Generating dimensions...
Saving parquet files...
Saving optimized parquet files...

--- Manifest ---
{
  "created_at_utc": "2026-06-13T18:42:18.593382+00:00",
  "group_id": 4,
  "variant": {
    "name": "Application logs",
    "feature": "status codes",
    "stress": "selective filters and string columns"
  },
  "scale": "large",
  "rows": 50000000,
  "run_seed": 289088503403094465115627696573718415941,
  "paths": {
    "events": "..\\data\\phase2_26L\\group_04\\events.parquet",
    "events_partitioned": "..\\data\\phase2_26L\\group_04\\events_partitioned",
    "events_optimized": "..\\data\\phase2_26L\\group_04\\events_optimized.parquet",
    "dimension": "..\\data\\phase2_26L\\group_04\\dimension.parquet"
  },
  "environment": {
    "python": "3.12.10",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 20,
    "ram_gib": 31.77
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [8]:
print(f"Events table row count: {events.height:,}")
print(f"Dimension table row count: {dimension.height:,}\n")

print("--- Schema of the 'events' table ---")
for col_name, dtype in events.schema.items():
    print(f"{col_name}: {dtype}")

print("\n--- Missing values---")
display(events.null_count())

print("\n--- HTTP codes distribution ---")
display(events.group_by("status_code").len().sort("status_code"))

print("\n--- Log levels distribution ---")
display(events.group_by("log_level").len().sort("len", descending=True))

print("\n--- Data sample (first 3 rows) ---")
display(events.head(3))

print("\n--- Dimension table Microservices directory ---")
display(dimension)

Events table row count: 50,000,000
Dimension table row count: 6

--- Schema of the 'events' table ---
event_id: Int64
session_id: Int64
event_ts: Datetime(time_unit='us', time_zone=None)
log_level: String
country: String
device: String
tags: List(String)
event_date: Date
status_code: Int32
endpoint: String
http_method: String
response_time_ms: Float32
payload_size_bytes: Int32

--- Missing values---


event_id,session_id,event_ts,log_level,country,device,tags,event_date,status_code,endpoint,http_method,response_time_ms,payload_size_bytes
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,2500000,0,0,0,0,0,0,0



--- HTTP codes distribution ---


status_code,len
i32,u32
200,32502338
201,2497744
400,2497479
401,2502296
403,1000132
404,7500167
500,500364
502,500254
503,499226



--- Log levels distribution ---


log_level,len
str,u32
"""INFO""",16666716
"""FATAL""",8334721
"""ERROR""",8333637
"""DEBUG""",8333429
"""WARN""",8331497



--- Data sample (first 3 rows) ---


event_id,session_id,event_ts,log_level,country,device,tags,event_date,status_code,endpoint,http_method,response_time_ms,payload_size_bytes
i64,i64,datetime[μs],str,str,str,list[str],date,i32,str,str,f32,i32
1,2308,2026-02-02 13:56:41,"""INFO""","""PL""","""desktop""","[""etl""]",2026-02-02,201,"""/api/checkout""","""POST""",1498.675049,65230
2,543,2026-02-07 20:23:58,"""FATAL""","""DE""","""mobile""","[""etl""]",2026-02-07,200,"""/api/cart""","""GET""",2058.574951,26160
3,3304,2026-02-07 14:53:43,"""WARN""","""DE""","""mobile""","[""security"", ""spark""]",2026-02-07,200,"""/api/products""","""GET""",663.950012,13580



--- Dimension table Microservices directory ---


endpoint,microservice,team_owner,sla_timeout_ms
str,str,str,i64
"""/api/users""","""user-service""","""team-alpha""",200
"""/api/products""","""catalog-service""","""team-beta""",300
"""/api/cart""","""order-service""","""team-gamma""",150
"""/api/checkout""","""payment-service""","""team-gamma""",500
"""/api/search""","""search-service""","""team-beta""",400
"""/api/auth""","""auth-service""","""team-alpha""",100


This dataset aligns with the "Application logs" domain profile and provides ground for performance benchmarking:

High Selectivity & Data Skew: The status_code column is intentionally skewed e.g., millions of 200/INFO logs, but very rare 50x errors. This is ideal for testing the engines abilities to perform selective filtering and predicate pushdown.

String Operations: We have multiple high-cardinality text columns (endpoint, http_method, log_level). This will expose the performance differences between the Pandas default object backend, the PyArrow string backend, and DuckDB's native string handling.

Join Viability: The presence of the endpoint column in both the massive events table and the small dimension table allows us to benchmark dimension table joins.

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [9]:
# TODO: Implement or adapt benchmark helpers before collecting final results.
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []
reference_results = {}


def run_benchmark(library_engine, mode, query_name, query_func, notes="", data_format="parquet", layout="default",
                  rows=N_ROWS, iterations=5):
    times = []
    mem_usages = []
    process = psutil.Process(os.getpid())

    for _ in range(iterations):
        gc.collect()
        mem_before = process.memory_info().rss / (1024 * 1024)

        start_time = time.perf_counter()
        result = query_func()
        end_time = time.perf_counter()

        mem_after = process.memory_info().rss / (1024 * 1024)

        times.append(end_time - start_time)
        mem_usages.append(max(0, mem_after - mem_before))

    median_time = np.median(times)
    avg_peak_mem = np.mean(mem_usages)

    try:
        res_len = result.shape[0] if hasattr(result, "shape") else len(result)
    except:
        res_len = "N/A"

    result_status = "Success"
    if res_len != "N/A":
        if query_name not in reference_results:
            reference_results[query_name] = res_len
            result_status = f"Reference (len: {res_len})"
        elif reference_results[query_name] != res_len:
            result_status = f"Mismatch! (Expected {reference_results[query_name]}, got {res_len})"

    benchmark_results.append({
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(avg_peak_mem, 2),
        "input_size_mb": None,
        "result_check": result_status,
        "notes": notes
    })

    print(
        f"{library_engine[:8]:8} | {mode[:9]:9} | Time: {median_time:.4f} s | Memory: ~{avg_peak_mem:7.2f} MB | {result_status}")

    return result

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [10]:
# TODO: Define your three query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

### Query 1: Selective Filter & Aggregation
**Description:** Find the frequency of critical server errors (`status_code IN (500, 502, 503)`) grouped by `endpoint`.
* **What does this query test?** It tests *selective filter plus aggregation* and is highly sensitive to *predicate pushdown* and *row-group pruning*.
* **Which library/engine do you expect to perform best?** DuckDB and Polars (Lazy/Streaming). They will push the filter down to the Parquet reader and skip scanning unneeded data blocks.
* **Which library/engine may use the most memory?** Pandas (default eager mode). It lacks native pushdown and will load all 2 million rows into Python memory before applying the filter.
* **Which physical layout should help, if any?** Sorting the Parquet file by `status_code` before writing (which we did in `events_optimized.parquet`). This puts all 50x errors in the same row groups, allowing engines to completely skip loading most of the file (the 200 OK codes).

---

### Query 2: Dimension Join & Aggregation
**Description:** Calculate the average response time (`response_time_ms`) for each internal team (`team_owner`) by joining the events table with the microservices dimension table on `endpoint`.
* **What does this query test?** It tests a *join with a dimension table* on a string column, followed by an aggregation. It also heavily relies on *column pruning*.
* **Which library/engine do you expect to perform best?** DuckDB due to its highly optimized vectorized execution and hash joins. Polars should be a very close second.
* **Which library/engine may use the most memory?** Pandas. Joining on string columns represented as Python objects is notoriously memory-intensive. The PyArrow-backed Pandas should perform noticeably better here.
* **Which physical layout should help, if any?** The native columnar layout of Parquet is crucial here, as it allows engines to only read `endpoint` and `response_time_ms` from the massive events table, completely ignoring heavy columns like `tags` or `device`.

---

### Query 3: Array/List Explode & Top-K
**Description:** Identify the top 5 most frequently occurring tags in logs where `log_level == 'ERROR'`.
* **What does this query test?** It covers *list/tag explode*, *top-k or sorting*, and selective filtering.
* **Which library/engine do you expect to perform best?** Polars. It has native, optimized support for array data types. DuckDB using `UNNEST` will also be very efficient.
* **Which library/engine may use the most memory?** Pandas. Its `.explode()` function handles lists very poorly, creating massive data duplication in memory. Peak memory usage will likely spike significantly compared to other engines. PySpark might also show some overhead due to JVM object instantiation during the explode phase.
* **Which physical layout should help, if any?** Since we filter by `log_level`, sorting the data by `log_level` during write would help via row-group pruning. Partitioning by `event_date` would not help here since date is not part of the query predicate.

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


<!-- spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

benchmark_results = []

def run_benchmark(library_engine, mode, query_name, query_func, data_format="parquet", layout="default", rows=N_ROWS, iterations=5):
    times = []
    mem_usages = []
    process = psutil.Process(os.getpid())

    for _ in range(iterations):
        gc.collect()
        mem_before = process.memory_info().rss / (1024 * 1024)

        start_time = time.perf_counter()
        result = query_func()
        end_time = time.perf_counter()

        mem_after = process.memory_info().rss / (1024 * 1024)
        times.append(end_time - start_time)
        mem_usages.append(max(0, mem_after - mem_before))

    median_time = np.median(times)
    avg_peak_mem = np.mean(mem_usages)

    benchmark_results.append({
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(avg_peak_mem, 2),
        "result_check": "Success"
    })
    print(f"✅ {library_engine:10} | {mode:10} | Czas: {median_time:.4f} s | Pamięć RAM: ~{avg_peak_mem:.2f} MB")
    return result

print("--- Benchmark Zapytania 1: Wyszukiwanie błędów serwera (500, 502, 503) ---\n")

def q1_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
res_pd_arr = run_benchmark("Pandas 3.0", "pyarrow", "Q1: 50x Errors", q1_pandas_pyarrow)

def q1_polars_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len()
res_pl_eager = run_benchmark("Polars", "eager", "Q1: 50x Errors", q1_polars_eager)

def q1_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect()
res_pl_lazy = run_benchmark("Polars", "lazy", "Q1: 50x Errors", q1_polars_lazy)

def q1_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect(engine="streaming")
res_pl_stream = run_benchmark("Polars", "streaming", "Q1: 50x Errors", q1_polars_streaming)

def q1_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
res_pd_def = run_benchmark("Pandas 3.0", "default", "Q1: 50x Errors", q1_pandas_default)

def q1_duckdb():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
res_duck = run_benchmark("DuckDB", "sql", "Q1: 50x Errors", q1_duckdb)

def q1_pyspark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(df.status_code.isin([500, 502, 503])).groupBy("endpoint").count().collect()
res_spark = run_benchmark("PySpark", "local", "Q1: 50x Errors", q1_pyspark)

display(pd.DataFrame(benchmark_results))

import pyspark.sql.functions as F

print("--- Benchmark Zapytania 2: JOIN z tabelą wymiarów i agregacja ---")
print("Opis: Łączymy logi z mikroserwisami po 'endpoint', a następnie liczymy średni czas odpowiedzi dla 'team_owner'.\n")

def q2_pandas_default():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)
    joined = events.merge(dim, on='endpoint', how='inner')
    return joined.groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas 3.0", "default", "Q2: Dim Join & Avg", q2_pandas_default)

def q2_pandas_pyarrow():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
    joined = events.merge(dim, on='endpoint', how='inner')
    return joined.groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas 3.0", "pyarrow", "Q2: Dim Join & Avg", q2_pandas_pyarrow)

def q2_polars_eager():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean())
run_benchmark("Polars", "eager", "Q2: Dim Join & Avg", q2_polars_eager)

def q2_polars_lazy():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect()
run_benchmark("Polars", "lazy", "Q2: Dim Join & Avg", q2_polars_lazy)

def q2_polars_streaming():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q2: Dim Join & Avg", q2_polars_streaming)

def q2_duckdb():
    query = f"""
        SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
        GROUP BY d.team_owner
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q2: Dim Join & Avg", q2_duckdb)

def q2_pyspark():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()
run_benchmark("PySpark", "local", "Q2: Dim Join & Avg", q2_pyspark)


print("\n--- Benchmark Zapytania 3: Array Explode i Top-K ---")
print("Opis: Szukamy logów 'ERROR', rozbijamy listy tagów na pojedyncze wiersze i znajdujemy 5 najczęstszych tagów.\n")

def q3_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas 3.0", "default", "Q3: Explode & Top-K", q3_pandas_default)

def q3_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas 3.0", "pyarrow", "Q3: Explode & Top-K", q3_pandas_pyarrow)

def q3_polars_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5)
run_benchmark("Polars", "eager", "Q3: Explode & Top-K", q3_polars_eager)

def q3_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect()
run_benchmark("Polars", "lazy", "Q3: Explode & Top-K", q3_polars_lazy)

def q3_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q3: Explode & Top-K", q3_polars_streaming)

def q3_duckdb():
    query = f"""
        SELECT tag, COUNT(*) as count
        FROM (
            SELECT UNNEST(tags) as tag
            FROM read_parquet('{EVENTS_PATH}')
            WHERE log_level = 'ERROR'
        )
        GROUP BY tag
        ORDER BY count DESC
        LIMIT 5
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q3: Explode & Top-K", q3_duckdb)

def q3_pyspark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(df.log_level == 'ERROR').withColumn("tag", F.explode("tags")).groupBy("tag").count().orderBy(F.col("count").desc()).limit(5).collect()
run_benchmark("PySpark", "local", "Q3: Explode & Top-K", q3_pyspark)

print("\n--- ZESTAWIENIE WYNIKÓW ---")
final_results_df = pd.DataFrame(benchmark_results)
display(final_results_df) -->

In [11]:
spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("PySpark version:", spark.version)

PySpark version: 4.1.2


In [12]:
print("--- Running benchmarks for PANDAS 3.0 ---\n")


def q1_pd_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()


run_benchmark("Pandas", "default", "Q1: Filter", q1_pd_default, notes="Backend: NumPy")


def q1_pd_arrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()


run_benchmark("Pandas", "pyarrow", "Q1: Filter", q1_pd_arrow, notes="Backend: PyArrow")


def q2_pd_default():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)
    return events.merge(dim, on='endpoint', how='inner').groupby('team_owner')['response_time_ms'].mean()


run_benchmark("Pandas", "default", "Q2: Join", q2_pd_default, notes="Backend: NumPy")


def q2_pd_arrow():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return events.merge(dim, on='endpoint', how='inner').groupby('team_owner')['response_time_ms'].mean()


run_benchmark("Pandas", "pyarrow", "Q2: Join", q2_pd_arrow, notes="Backend: PyArrow")


def q3_pd_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)


run_benchmark("Pandas", "default", "Q3: Explode", q3_pd_default, notes="Backend: NumPy")


def q3_pd_arrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)


run_benchmark("Pandas", "pyarrow", "Q3: Explode", q3_pd_arrow, notes="Backend: PyArrow")

--- Running benchmarks for PANDAS 3.0 ---

Pandas   | default   | Time: 22.0452 s | Memory: ~1919.75 MB | Reference (len: 6)
Pandas   | pyarrow   | Time: 5.5171 s | Memory: ~ 172.69 MB | Success
Pandas   | default   | Time: 32.8396 s | Memory: ~ 176.92 MB | Reference (len: 3)
Pandas   | pyarrow   | Time: 13.8675 s | Memory: ~ 930.81 MB | Success
Pandas   | default   | Time: 34.1880 s | Memory: ~1104.23 MB | Reference (len: 5)
Pandas   | pyarrow   | Time: 8.8899 s | Memory: ~ 627.39 MB | Success


tags
polars      1853556
sql         1852508
ai          1852492
security    1852313
duckdb      1852099
Name: count, dtype: int64[pyarrow]

In [13]:
print("\n--- Running benchmarks for POLARS ---\n")


def q1_pl_eager():
    return pl.read_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len()


run_benchmark("Polars", "eager", "Q1: Filter", q1_pl_eager)


def q1_pl_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by(
        'endpoint').len().collect()


run_benchmark("Polars", "lazy", "Q1: Filter", q1_pl_lazy)


def q1_pl_stream():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by(
        'endpoint').len().collect(engine="streaming")


run_benchmark("Polars", "streaming", "Q1: Filter", q1_pl_stream)


def q2_pl_eager():
    e = pl.read_parquet(EVENTS_PATH)
    d = pl.read_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean())


run_benchmark("Polars", "eager", "Q2: Join", q2_pl_eager)


def q2_pl_lazy():
    e = pl.scan_parquet(EVENTS_PATH)
    d = pl.scan_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect()


run_benchmark("Polars", "lazy", "Q2: Join", q2_pl_lazy)


def q2_pl_stream():
    e = pl.scan_parquet(EVENTS_PATH)
    d = pl.scan_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect(
        engine="streaming")


run_benchmark("Polars", "streaming", "Q2: Join", q2_pl_stream)


def q3_pl_eager():
    return pl.read_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by(
        'tags').len().sort('len', descending=True).head(5)


run_benchmark("Polars", "eager", "Q3: Explode", q3_pl_eager)


def q3_pl_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by(
        'tags').len().sort('len', descending=True).head(5).collect()


run_benchmark("Polars", "lazy", "Q3: Explode", q3_pl_lazy)


def q3_pl_stream():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by(
        'tags').len().sort('len', descending=True).head(5).collect(engine="streaming")


run_benchmark("Polars", "streaming", "Q3: Explode", q3_pl_stream)


--- Running benchmarks for POLARS ---

Polars   | eager     | Time: 2.0230 s | Memory: ~1964.36 MB | Success
Polars   | lazy      | Time: 0.0677 s | Memory: ~   0.00 MB | Success
Polars   | streaming | Time: 0.0482 s | Memory: ~  10.72 MB | Success
Polars   | eager     | Time: 13.4962 s | Memory: ~1657.53 MB | Success
Polars   | lazy      | Time: 1.1972 s | Memory: ~2248.90 MB | Success
Polars   | streaming | Time: 0.6305 s | Memory: ~  46.67 MB | Success
Polars   | eager     | Time: 2.8369 s | Memory: ~ 708.76 MB | Success
Polars   | lazy      | Time: 1.0815 s | Memory: ~  11.85 MB | Success
Polars   | streaming | Time: 0.7232 s | Memory: ~ 117.26 MB | Success


tags,len
str,u32
"""polars""",1853556
"""sql""",1852508
"""ai""",1852492
"""security""",1852313
"""duckdb""",1852099


In [14]:
print("\n--- Running benchmark for DUCKDB ---\n")


def q1_duckdb():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "sql", "Q1: Filter", q1_duckdb)


def q2_duckdb():
    query = f"""
        SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
        GROUP BY d.team_owner
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "sql", "Q2: Join", q2_duckdb)


def q3_duckdb():
    query = f"""
        SELECT tag, COUNT(*) as count
        FROM (
            SELECT UNNEST(tags) as tag
            FROM read_parquet('{EVENTS_PATH}')
            WHERE log_level = 'ERROR'
        )
        GROUP BY tag
        ORDER BY count DESC
        LIMIT 5
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "sql", "Q3: Explode", q3_duckdb)


--- Running benchmark for DUCKDB ---

DuckDB   | sql       | Time: 0.1778 s | Memory: ~  11.47 MB | Success
DuckDB   | sql       | Time: 0.4900 s | Memory: ~   3.61 MB | Success
DuckDB   | sql       | Time: 0.6870 s | Memory: ~   1.21 MB | Success


,tag,count
0,polars,1853556
1,sql,1852508
2,ai,1852492
3,security,1852313
4,duckdb,1852099


In [15]:
import pyspark.sql.functions as F

print("\n--- Running benchmarks for PYSPARK (LOCAL) ---\n")


def q1_spark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(F.col("status_code").isin([500, 502, 503])).groupBy("endpoint").count().collect()


run_benchmark("PySpark", "local", "Q1: Filter", q1_spark)


def q2_spark():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()


run_benchmark("PySpark", "local", "Q2: Join", q2_spark)


def q3_spark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(F.col("log_level") == "ERROR").withColumn("tag", F.explode("tags")).groupBy("tag").count().orderBy(
        F.col("count").desc()).limit(5).collect()


run_benchmark("PySpark", "local", "Q3: Explode", q3_spark)


--- Running benchmarks for PYSPARK (LOCAL) ---

PySpark  | local     | Time: 0.9332 s | Memory: ~   0.00 MB | Success
PySpark  | local     | Time: 3.4633 s | Memory: ~   0.00 MB | Success
PySpark  | local     | Time: 1.7031 s | Memory: ~   0.00 MB | Success


[Row(tag='polars', count=1853556),
 Row(tag='sql', count=1852508),
 Row(tag='ai', count=1852492),
 Row(tag='security', count=1852313),
 Row(tag='duckdb', count=1852099)]

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [16]:

print("--- Preparing csv ---")
pl.read_parquet(EVENTS_PATH).select(["status_code", "endpoint"]).write_csv(CSV_EVENTS_PATH)

print("--- File benchmark(DuckDb) ---")


def q1_duckdb_default():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "Parquet Default", "Q1: Layout Test", q1_duckdb_default, notes="Random rows structer")


def q1_duckdb_optimized():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{OPTIMIZED_EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "Parquet Opt.", "Q1: Layout Test", q1_duckdb_optimized, notes="Sorted status codes")


def q1_duckdb_csv():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_csv_auto('{CSV_EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()


run_benchmark("DuckDB", "CSV Baseline", "Q1: Layout Test", q1_duckdb_csv, notes="2 columns without compression")

print("\n--- ETAP 3: Dowody fizyczne (Rozmiar i IO) ---")
size_default = os.path.getsize(EVENTS_PATH) / (1024 * 1024)
size_opt = os.path.getsize(OPTIMIZED_EVENTS_PATH) / (1024 * 1024)
size_csv = os.path.getsize(CSV_EVENTS_PATH) / (1024 * 1024)

print(f"Size on the deisk default parquet: {size_default:.2f} MB")
print(f"Size on the diiskOptimized parquet:              {size_opt:.2f} MB")
print(f"Csv only 2 rows): {size_csv:.2f} MB")

explain_plan = duckdb.sql(
    f"EXPLAIN SELECT endpoint FROM read_parquet('{OPTIMIZED_EVENTS_PATH}') WHERE status_code = 500").df()
print("\nDuckDB EXPLAIN plan 'Filters: status_code=500'):")
print(explain_plan['explain_value'].iloc[0])


--- Preparing csv ---
--- File benchmark(DuckDb) ---
DuckDB   | Parquet D | Time: 0.3315 s | Memory: ~  11.15 MB | Reference (len: 6)
DuckDB   | Parquet O | Time: 0.0519 s | Memory: ~   0.88 MB | Success
DuckDB   | CSV Basel | Time: 1.1038 s | Memory: ~   0.69 MB | Success

--- ETAP 3: Dowody fizyczne (Rozmiar i IO) ---
Size on the deisk default parquet: 992.46 MB
Size on the diiskOptimized parquet:              1048.09 MB
Csv only 2 rows): 755.02 MB

DuckDB EXPLAIN plan 'Filters: status_code=500'):
┌───────────────────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│   Projections: endpoint   │
│                           │
│          Filters:         │
│      status_code=500      │
│                           │
│      ~10,000,000 rows     │
└───────────────────────────┘



### Task 2.5 Explanation
In this experiment, we tested Query 1 (`status_code IN (500, 502, 503)`) across three different physical layouts. The results illustrate the differences in file architectures:

#### 1. Why is the Optimized Parquet the fastest? (0.0519 s vs 0.3315 s)
During the generation of the optimized file, we applied **sorting by the `status_code` column**. Because of this, the rare 50x errors (making up roughly ~3% of the data) were physically grouped together. The Parquet format stores metadata (`min` and `max` values) for each Row Group. The DuckDB engine reads this metadata, and upon seeing that a given group only contains codes from `200` to `404`, completely skips loading it from the disk.
The proof of this is the **Row Group Pruning** and **Predicate Pushdown** mechanism. This optimization reduced the execution time from 0.33s to just ~0.05s and drastically cut RAM usage from ~11.15 MB down to a mere ~0.88 MB.

#### 2. The Size Puzzle: Why does the CSV file take up less space (755.02 MB) than Parquet (992.46 MB)?
At first glance, this contradicts the power of Parquet compression, but we are comparing two completely different datasets here:
* **Parquet files (~992 MB - 1048 MB)** contain **all 13 columns**, including very heavy data such as dates, strings, and nested lists of tags for 50 million rows.
* **The CSV file (755.02 MB)** is our "Negative Baseline" created specifically for this query – it contains **only 2 columns** (`status_code` and `endpoint`).
If we had saved all 13 columns as plain text to a CSV, the file would weigh several gigabytes. The ~1 GB result for the full 50M rows data in Parquet format is actually an excellent compression level (zstd).

#### 3. Why is the optimized Parquet slightly heavier than the default one? (1048.09 MB vs 992.46 MB)
The optimized file weighs 1048 MB (vs. 992 MB for the default). This is due to the fact that we enforced a smaller `row_group_size=100_000`. For 50 million rows, this creates 500 separate blocks. A larger number of smaller blocks means that a larger amount of metadata (`min/max` statistics) must be stored in the file footer. Additionally, sorting the data by one column (`status_code`) "scatters" the natural order of other columns, which slightly degrades the efficiency of the compression algorithm for the rest of the file.

#### 4. Why is the CSV file so drastically slow? (1.1038 s)
Even though the CSV file has only two columns and a smaller footprint on disk, it was over 20 times slower than the optimized Parquet. A CSV file is a flat text file devoid of columnar structure and metadata. DuckDB had to perform a **Full Table Scan** – loading and parsing 755 MB of text character by character to find the 50x errors.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [17]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# TODO 3.1: Implement Polars execution-mode experiments.

print("--- Task 3.1: Polars Execution Modes (Heavy Filter) ---\n")

SINK_OUTPUT_PATH = OUTPUT_DIR / "task31_sink_output.parquet"

COLUMNS_TO_SELECT = ["event_id", "session_id", "event_ts", "endpoint", "tags", "payload_size_bytes"]


def t31_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT)


def t31_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).collect()


def t31_stream_collect():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).collect(
        engine="streaming")


def t31_stream_sink():
    pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).sink_parquet(
        SINK_OUTPUT_PATH)
    return None


run_benchmark("Polars", "eager", "T3.1: Heavy", t31_eager, notes="Full RAM load")
run_benchmark("Polars", "lazy", "T3.1: Heavy", t31_lazy, notes="Optimized read, RAM collect")
run_benchmark("Polars", "stream_coll", "T3.1: Heavy", t31_stream_collect, notes="Batched read, RAM collect")
run_benchmark("Polars", "stream_sink", "T3.1: Heavy", t31_stream_sink, notes="Batched read & write to disk")

if SINK_OUTPUT_PATH.exists():
    sink_size = os.path.getsize(SINK_OUTPUT_PATH) / (1024 * 1024)
    print(f"\nSize of results file (sink_parquet): {sink_size:.2f} MB")

--- Task 3.1: Polars Execution Modes (Heavy Filter) ---

Polars   | eager     | Time: 1.7086 s | Memory: ~ 305.82 MB | Reference (len: 16666716)
Polars   | lazy      | Time: 1.1128 s | Memory: ~ 268.30 MB | Success
Polars   | stream_co | Time: 1.1090 s | Memory: ~ 508.65 MB | Success
Polars   | stream_si | Time: 2.3103 s | Memory: ~ 233.72 MB | Success

Size of results file (sink_parquet): 246.04 MB


### Task 3.1 Conclusion

In this experiment, we tested a "Heavy Filter" operation (which returned a massive subset of over **16.6 million rows** from the initial 50 million) to demonstrate the critical differences in RAM management across various Polars execution modes:

1. **Eager Mode (`read_parquet`):** This is the most memory-intensive approach. It loads the entire 50-million row file into RAM before applying any filters or dropping unneeded columns. Although the recorded memory delta was ~305 MB, this mode forces the entire dataset to reside in memory simultaneously. For larger datasets, this will inevitably and quickly lead to an Out-Of-Memory (OOM) error.
2. **Lazy Mode (`scan_parquet` + `collect`):** We can observe the power of the query optimizer (Time: 1.11 s | Mem: ~268 MB). The engine applies *Predicate Pushdown* and *Column Pruning* at the disk-read level. However, because we use `.collect()`, the engine is still forced to build and materialize the final table of 16.6 million rows entirely in Python's RAM at the very end.
3. **Streaming Collect:** Even though the streaming engine processes the data in smaller batches to minimize computational overhead, calling `.collect(engine="streaming")` ultimately forces the system to allocate the final, massive result set in RAM. This is why we observed the highest memory spike in this mode (~508 MB) — the system had to hold all 16.6 million processed rows in memory at once.
4. **Streaming Sink (`sink_parquet`):** This represents true *Out-of-Core* processing. Polars loads a chunk of data from the disk, filters it, and immediately writes it to the target file (generating a 246 MB Parquet file), releasing the memory before fetching the next chunk. While the memory usage showed ~233 MB (likely due to disk write buffers and Jupyter's background memory retention), this is the only safe method for processing datasets that exceed the machine's physical RAM, as its memory footprint does not scale linearly with the total dataset size.


###  Why did `stream_collect` use more memory than `eager` mode?

At first glance, it might seem counterintuitive that the streaming collect mode (~508 MB) consumed more peak memory than the eager mode (~305 MB). This is not a measurement error, but a direct result of how the Polars engine operates under the hood:

1. **Chunk Concatenation Overhead:** In streaming mode, Polars processes data in small, manageable batches. However, when `.collect()` is called at the end of the query, the engine is forced to merge hundreds of these processed batches into a single, contiguous DataFrame (16.6 million rows). During this concatenation phase, both the individual chunks and the newly allocated final table must briefly coexist in RAM, causing a massive temporary spike in memory usage.
2. **Thread Buffers & Queues:** The streaming engine is highly multi-threaded. On a machine with 20 logical cores, Polars maintains separate data queues and memory buffers for each thread to allow asynchronous processing. Holding all these active buffers for a massive result set artificially inflates the peak memory (RSS) tracked by the operating system.
3. **Memory Mapping (mmap) in Eager Mode:** The `read_parquet` function (used in eager mode) relies heavily on memory mapping. This allows the OS to pretend the entire file is in RAM while only physically loading the pages currently being accessed, seamlessly evicting unused ones. In contrast, the streaming engine combined with `.collect()` forces hard heap allocations for the final materialized result.

---



#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [33]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


POLARS_LIMITATION_SCENARIO = """
**Vertical Scaling Bottleneck and lack of Distributed Fault Tolerance.**

While Polars on a single machine offers phenomenal performance and can process data exceeding RAM capacity (Out-of-Core) using streaming disk writes (`sink_parquet`), it hits a hard architectural barrier when attempting to materialize large results in Python. Since Polars is a Single-Node engine, any operation ending with `.collect()` forces the entire DataFrame to be built in the RAM of that single machine. The lack of a cluster manager also means there are no fault tolerance mechanisms—unlike Apache Spark, an Out-of-Memory (OOM) error immediately and permanently kills the entire process without any attempt to retry or redistribute the workload.
"""

POLARS_LIMITATION_EVIDENCE = """
Evidence from the **Task 3.1 (Heavy Filter)** experiment perfectly illustrates this problem. The query filtered 50 million rows down to approximately 16.6 million rows. We observed that calling `.collect(engine="streaming")` forced the simultaneous allocation of the final result in memory, resulting in the highest peak RAM usage of **~508 MB**.

Extrapolating this result: if we applied this exact same code to a real Big Data scale (e.g., 5 billion input rows), the materialized result alone would consume over **50 GB** of memory. On a standard machine (like ours with ~32 GB RAM), using `.collect()` would result in a critical Out-Of-Memory error, despite using the streaming engine under the hood.

**Why Apache Spark would perform better in this scenario:**
In an environment where the result set is too large for a single machine or there is a risk of node failure, a distributed cluster (e.g., Dataproc) becomes necessary. Apache Spark inherently distributes work (Partitions) across multiple machines. If it runs out of RAM, Spark safely spills excess data to the cluster's disks.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

**Polars limitation scenario**

**Vertical Scaling Bottleneck and lack of Distributed Fault Tolerance.**

While Polars on a single machine offers phenomenal performance and can process data exceeding RAM capacity (Out-of-Core) using streaming disk writes (`sink_parquet`), it hits a hard architectural barrier when attempting to materialize large results in Python. Since Polars is a Single-Node engine, any operation ending with `.collect()` forces the entire DataFrame to be built in the RAM of that single machine. The lack of a cluster manager also means there are no fault tolerance mechanisms—unlike Apache Spark, an Out-of-Memory (OOM) error immediately and permanently kills the entire process without any attempt to retry or redistribute the workload.

**Evidence**

Evidence from the **Task 3.1 (Heavy Filter)** experiment perfectly illustrates this problem. The query filtered 50 million rows down to approximately 16.6 million rows. We observed that calling `.collect(engine="streaming")` forced the simultaneous allocation of the final result in memory, resulting in the highest peak RAM usage of **~508 MB**.

Extrapolating this result: if we applied this exact same code to a real Big Data scale (e.g., 5 billion input rows), the materialized result alone would consume over **50 GB** of memory. On a standard machine (like ours with ~32 GB RAM), using `.collect()` would result in a critical Out-Of-Memory error, despite using the streaming engine under the hood.

**Why Apache Spark would perform better in this scenario:**
In an environment where the result set is too large for a single machine or there is a risk of node failure, a distributed cluster (e.g., Dataproc) becomes necessary. Apache Spark inherently distributes work (Partitions) across multiple machines. If it runs out of RAM, Spark safely spills excess data to the cluster's disks.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [34]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


DECISION_BOUNDARY = """
Based on our measurements, we recommend migrating from Single-Node tools (Polars/DuckDB) to a distributed engine (Apache Spark) ** when at least one of the following conditions is met**:

1. **Materialization Bottleneck:** The final result set (or intermediate state for complex `JOIN` / `GROUP BY` operations) exceeds the physical RAM capacity of a single machine (e.g., >32 GB or >64 GB), and business logic requires materializing this data in memory (the equivalent of `.collect()`) rather than writing it directly to disk (Out-of-Core/Sink).
2. **Storage Bottleneck (Storage-Bound):** The raw input data exceeds the capacity of fast local disks on a single server and must be stored on a distributed file system  like Google Cloud Storage, where a Spark cluster can leverage *Data Locality*.
3. **Time Complexity and Fault Tolerance:** Processing time on a single node (even using `sink_parquet`) becomes unacceptably long for ETL pipelines (e.g., taking several hours). In such cases, Spark not only distributes the workload across multiple machines to speed up execution but also guarantees fault tolerance. If one worker node crashes 5 hours into a job, Spark will recompute only its lost partitions, whereas Polars on a single machine would fail completely, forcing a restart from scratch.
"""

DECISION_EVIDENCE = """

1. **Spark's Overhead on Medium-Sized Data (Task 2):** Our dataset was approximately 1 GB (50 million rows). Single-node engines like DuckDB and Polars (Lazy) executed a selective filter (Query 1) in mere fractions of a second (**~0.06s to 0.17s**). Meanwhile, PySpark (Local) required **~0.93 seconds**. The gap widened in Query 2 (Join & Aggregation), where DuckDB finished in **~0.49s** while PySpark took **~3.46s**. This demonstrates that for datasets that fit comfortably in RAM, Spark loses out significantly due to the overhead of the Java Virtual Machine (JVM), task serialization, and DAG management. Using Spark for datasets in the lower gigabyte range is an inefficient use of resources.

2. **Linear RAM Scaling in Polars (Task 3.1):** In the `Heavy Filter` test, we forced Polars to filter and retain ~16.6 million rows in memory. This allocated **~268 MB** of RAM in Lazy mode and spiked to **~508 MB** during streaming collection. Extrapolating this linearly: if we were to process and materialize a dataset just 100 times larger (~1.6 billion resulting rows), the Polars engine would demand between **26 GB and 50 GB of RAM**. With real-world Big Data datasets, Polars would inevitably hit an OOM (Out-Of-Memory) error on any standard server during a `.collect()` call. In contrast, Spark would safely manage this by spilling the excess data to the cluster's disks (Disk Spilling/Shuffle).
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Based on our measurements, we recommend migrating from Single-Node tools (Polars/DuckDB) to a distributed engine (Apache Spark) ** when at least one of the following conditions is met**:

1. **Materialization Bottleneck:** The final result set (or intermediate state for complex `JOIN` / `GROUP BY` operations) exceeds the physical RAM capacity of a single machine (e.g., >32 GB or >64 GB), and business logic requires materializing this data in memory (the equivalent of `.collect()`) rather than writing it directly to disk (Out-of-Core/Sink).
2. **Storage Bottleneck (Storage-Bound):** The raw input data exceeds the capacity of fast local disks on a single server and must be stored on a distributed file system  like Google Cloud Storage, where a Spark cluster can leverage *Data Locality*.
3. **Time Complexity and Fault Tolerance:** Processing time on a single node (even using `sink_parquet`) becomes unacceptably long for ETL pipelines (e.g., taking several hours). In such cases, Spark not only distributes the workload across multiple machines to speed up execution but also guarantees fault tolerance. If one worker node crashes 5 hours into a job, Spark will recompute only its lost partitions, whereas Polars on a single machine would fail completely, forcing a restart from scratch.

**Evidence**

1. **Spark's Overhead on Medium-Sized Data (Task 2):** Our dataset was approximately 1 GB (50 million rows). Single-node engines like DuckDB and Polars (Lazy) executed a selective filter (Query 1) in mere fractions of a second (**~0.06s to 0.17s**). Meanwhile, PySpark (Local) required **~0.93 seconds**. The gap widened in Query 2 (Join & Aggregation), where DuckDB finished in **~0.49s** while PySpark took **~3.46s**. This demonstrates that for datasets that fit comfortably in RAM, Spark loses out significantly due to the overhead of the Java Virtual Machine (JVM), task serialization, and DAG management. Using Spark for datasets in the lower gigabyte range is an inefficient use of resources.

2. **Linear RAM Scaling in Polars (Task 3.1):** In the `Heavy Filter` test, we forced Polars to filter and retain ~16.6 million rows in memory. This allocated **~268 MB** of RAM in Lazy mode and spiked to **~508 MB** during streaming collection. Extrapolating this linearly: if we were to process and materialize a dataset just 100 times larger (~1.6 billion resulting rows), the Polars engine would demand between **26 GB and 50 GB of RAM**. With real-world Big Data datasets, Polars would inevitably hit an OOM (Out-Of-Memory) error on any standard server during a `.collect()` call. In contrast, Spark would safely manage this by spilling the excess data to the cluster's disks (Disk Spilling/Shuffle).

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [30]:
import psutil

print("--- Task 4: Thread and Core Scalability ---\n")

max_cores = psutil.cpu_count(logical=True)
cores_to_test = [1, 2, 4, 8, max_cores]

print(">> Testing ducks db scalability...")
for cores in cores_to_test:
    duckdb.sql(f"PRAGMA threads={cores}")


    def q2_duckdb_scale():
        query = f"""
            SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
            FROM read_parquet('{EVENTS_PATH}') e
            JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
            GROUP BY d.team_owner
        """
        return duckdb.sql(query).df()


    run_benchmark("DuckDB", f"{cores}_threads", "T4: Scale Q2", q2_duckdb_scale, notes=f"DuckDB threads: {cores}")

duckdb.sql(f"PRAGMA threads={max_cores}")

print("\n>> Testing PySpark scalability...")

try:
    spark.stop()
except:
    pass

for cores in cores_to_test:
    print(f"Running Spark master='local[{cores}]'...")
    spark_scale = (
        SparkSession.builder
        .appName(f"ScalabilityTest_{cores}")
        .master(f"local[{cores}]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )


    def q2_spark_scale():
        events = spark_scale.read.parquet(str(EVENTS_PATH))
        dim = spark_scale.read.parquet(str(DIMENSION_PATH))
        return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()


    run_benchmark("PySpark", f"local[{cores}]", "T4: Scale Q2", q2_spark_scale, notes=f"Spark local[{cores}]")

    spark_scale.stop()

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

--- Task 4: Thread and Core Scalability ---

>> Testing ducks db scalability...
DuckDB   | 1_threads | Time: 2.2365 s | Memory: ~   2.27 MB | Success
DuckDB   | 2_threads | Time: 1.1168 s | Memory: ~   1.11 MB | Success
DuckDB   | 4_threads | Time: 0.5764 s | Memory: ~   1.13 MB | Success
DuckDB   | 8_threads | Time: 0.3266 s | Memory: ~   9.53 MB | Success
DuckDB   | 20_thread | Time: 0.1907 s | Memory: ~   5.45 MB | Success

>> Testing PySpark scalability...
Running Spark master='local[1]'...
PySpark  | local[1]  | Time: 5.1032 s | Memory: ~   0.00 MB | Success
Running Spark master='local[2]'...
PySpark  | local[2]  | Time: 3.2586 s | Memory: ~   0.00 MB | Success
Running Spark master='local[4]'...
PySpark  | local[4]  | Time: 1.9863 s | Memory: ~   0.00 MB | Success
Running Spark master='local[8]'...
PySpark  | local[8]  | Time: 1.9139 s | Memory: ~   0.00 MB | Success
Running Spark master='local[20]'...
PySpark  | local[20] | Time: 1.4864 s | Memory: ~   0.00 MB | Success


### Task 4 Conclusions: Thread and Core Scalability

Based on our measurements for DuckDB and PySpark on a machine with 20 logical cores, we observed that while adding computational resources improves execution time, **it does not scale in a perfectly linear fashion indefinitely**. The two engines exhibited vastly different scaling behaviors due to their underlying architectures.

#### 1. DuckDB: Near-Linear Scaling with Low Overhead
DuckDB demonstrated phenomenal scalability. As we doubled the threads from 1 to 2, to 4, and to 8, the execution time almost perfectly halved at each step (dropping from **2.23s** to **1.11s**, **0.57s**, and **0.32s**).
* **Why it scales so well:** DuckDB is a lightweight, in-process C++ engine. It shares memory directly with Python and utilizes vectorized query execution without the need for complex cluster-like task scheduling.
* **The tapering effect:** Pushing it to 20 threads yielded **0.19s**. The scaling curve slightly flattens here because of hardware limits.

#### 2. PySpark (Local Mode): Severe Diminishing Returns
PySpark showed initial parallelization gains but quickly hit a hard bottleneck. Moving from `local[1]` (**5.10s**) to `local[2]` (**3.25s**) to `local[4]` (**1.98s**) yielded noticeable speedups. However, moving from 4 cores to 8 cores (**1.91s**) and even 20 cores (**1.48s**) provided minimal benefits.
* **Why scaling stops:** Spark is fundamentally designed for distributed clusters. In `local` mode, it still simulates a master-worker architecture. When dealing with a relatively "small" workload for 20 cores (finishing in under 2 seconds), the **Task Scheduling Overhead** and **JVM (Java Virtual Machine) Thread Coordination** become the dominant bottlenecks.
* **Shuffle Overhead:** Query 2 involves a `JOIN` and a `GROUP BY`, which forces a shuffle operation. Coordinating data exchange across 20 simulated executors on a single machine introduces significant synchronization delays.

#### 3. General Observation (Amdahl's Law)
Both engines eventually fall victim to **Amdahl's Law**, which states that the overall speedup of a system is limited by its strictly sequential parts. Tasks like reading Parquet metadata, initializing the query plan, and allocating initial memory cannot be perfectly parallelized. As we add more cores, the execution time of the parallel portion shrinks, leaving the fixed sequential overhead as the absolute floor limit for performance.

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [21]:
import time
import statistics
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

BUCKET_PATH = "gs://tbd-2026l-325157-data/phase2_large"
ITERATIONS = 5


def main():
    spark = SparkSession.builder.appName("TBDPhase2_Task5_Dataproc").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    events = spark.read.parquet(f"{BUCKET_PATH}/events_optimized.parquet")
    dim = spark.read.parquet(f"{BUCKET_PATH}/dimension.parquet")

    events.select("status_code", "endpoint", "log_level", "response_time_ms", "tags").limit(10).collect()
    dim.limit(10).collect()

    times_q1 = []
    events.filter(F.col("status_code").isin([500, 502, 503])).groupBy("endpoint").count().collect()
    for i in range(ITERATIONS):
        start = time.time()
        res = events.filter(F.col("status_code").isin([500, 502, 503])).groupBy("endpoint").count().collect()
        times_q1.append(time.time() - start)

    times_q2 = []
    events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()
    for i in range(ITERATIONS):
        start = time.time()
        res = events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()
        times_q2.append(time.time() - start)

    times_q3 = []
    events.filter(F.col("log_level") == "ERROR").withColumn("tag", F.explode("tags")).groupBy("tag").count().orderBy(
        F.col("count").desc()).limit(5).collect()
    for i in range(ITERATIONS):
        start = time.time()
        res = events.filter(F.col("log_level") == "ERROR").withColumn("tag", F.explode("tags")).groupBy(
            "tag").count().orderBy(F.col("count").desc()).limit(5).collect()
        times_q3.append(time.time() - start)

    print(f"Q1 Median: {statistics.median(times_q1):.4f} s")
    print(f"Q2 Median: {statistics.median(times_q2):.4f} s")
    print(f"Q3 Median: {statistics.median(times_q3):.4f} s")

    spark.stop()


if __name__ == "__main__":
    main()

### Task 5: PySpark Performance Comparison (Local vs. Dataproc)

**1. PySpark Execution Script**
The above script was submitted to the Dataproc cluster. It measures the execution time of three distinct queries with varying computational profiles across 5 iterations to obtain a stable median.

```python

```

**2. Hardware Configurations**
* **Local Setup**: 20 Logical Cores, 32 GB RAM.
* **Dataproc Default Setup**: Minimal configuration (2 workers), severely constrained by vCPU and RAM. Data read from GCS.
* **Dataproc Optimized Setup**: 1x `e2-standard-4` (Master), 2x `e2-standard-8` (Workers).
**3. Performance Results (Median times in seconds for 50M rows)**

| Query Type | Local Setup (20 Cores, 32GB) | Dataproc (Default Setup) | Dataproc (Optimized: 16 vCPU, 64GB) |
| :--- | :--- | :--- | :--- |
| **Q1 (Filter + GroupBy)** | **0.9332 s** | 4.3302 s | 1.9133 s |
| **Q2 (Join + Agg)** | **3.4633 s** | 18.7180 s | 4.6566 s |
| **Q3 (Explode + Top-K)** | **1.7031 s** | 17.9012 s | 3.8393 s |

**4. Conclusions & Execution Characteristics Analysis**

* **Network I/O & Distributed Overhead:** Interestingly, the local machine outperformed the optimized Dataproc cluster across all three queries. This clearly exposes a classic Big Data threshold concept: **Network and Scheduling Overhead**. Our dataset is ~1 GB (50 million rows), which easily fits into the 32 GB RAM of the local machine. Reading 1 GB from a local SSD is almost instantaneous. In contrast, the Dataproc cluster must fetch Parquet blocks from Google Cloud Storage (`gs://`) over the network, drastically slowing down the baseline read operations. Furthermore, task serialization and DAG coordination across multiple physical nodes introduce latency that simply does not exist on a single machine.
* **Shuffle Mechanics & Q2 (Join + Agg):** Query 2 forces a Data Shuffle due to the `join` and `groupBy` operations. The default Dataproc cluster struggled immensely (18.7 seconds). The optimized cluster, with 64 GB of distributed RAM, handled the shuffle entirely **in-memory** (4.6 seconds). However, the local setup still won (3.4 seconds) because shuffling data between local CPU threads via shared memory is vastly faster than exchanging data between physical worker nodes over a network.
* **CPU Scalability & Partitioning (Q3 Explode):** The `explode()` function multiplies the number of rows processed in memory. The default Dataproc cluster was heavily CPU-bound and memory-starved, leading to high task serialization overhead (17.9s). The optimized cluster's 16 vCPUs allowed it to process more partitions concurrently, dropping the time to 3.8s. Yet, the local 20-core setup easily absorbed the expanded partition sizes in local memory, processing the top-K aggregation without distributed network overhead.
* **Final Verdict:** For a 1 GB dataset, setting up a distributed cluster yields negative performance returns due to network and distributed scheduling overhead. Dataproc's scaling advantages would only become visible on much larger datasets (e.g., 100+ GB) where the local machine's 32 GB RAM would trigger severe Out-of-Memory errors or massive OS disk swapping.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [8]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
Query 3 (Array Explode & Top-K) best exposes the paradigm differences between DataFrame and SQL engines. In DataFrame APIs (Pandas, Polars, PySpark), expanding an array into rows is typically handled via a procedural .explode() method chained seamlessly into group-by operations. In contrast, SQL engines like DuckDB require a more declarative approach using the UNNEST() function, often forcing the user to wrap the explosion step inside a subquery or a Common Table Expression (CTE) before aggregating the results.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

Query 3 (Array Explode & Top-K) best exposes the paradigm differences between DataFrame and SQL engines. In DataFrame APIs (Pandas, Polars, PySpark), expanding an array into rows is typically handled via a procedural .explode() method chained seamlessly into group-by operations. In contrast, SQL engines like DuckDB require a more declarative approach using the UNNEST() function, often forcing the user to wrap the explosion step inside a subquery or a Common Table Expression (CTE) before aggregating the results.

In [7]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
Query 3 (Explode & Top-K) and the Task 3.1 scenario (Heavy Filter) are the most memory-sensitive. The .explode() operation physically duplicates row data for every element in the array. In Pandas eager execution, this causes massive memory spikes due to Python object duplication. Furthermore, as seen in Task 3.1, any query that returns a massive result set is highly memory-sensitive if the engine is forced to materialize the final output in RAM (using .collect()).
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

Query 3 (Explode & Top-K) and the Task 3.1 scenario (Heavy Filter) are the most memory-sensitive. The .explode() operation physically duplicates row data for every element in the array. In Pandas eager execution, this causes massive memory spikes due to Python object duplication. Furthermore, as seen in Task 3.1, any query that returns a massive result set is highly memory-sensitive if the engine is forced to materialize the final output in RAM (using .collect()).

In [6]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
Yes, lazy execution drastically reduced the amount of data read from disk, but it did not change the amount of data materialized at the end. By using pl.scan_parquet(), Polars' query optimizer applied Predicate Pushdown and Column Pruning—it only read the strictly required columns from the disk and skipped entire row groups using Parquet metadata. However, because the queries ended with .collect(), the final result set was still fully materialized in Python's memory.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

Yes, lazy execution drastically reduced the amount of data read from disk, but it did not change the amount of data materialized at the end. By using pl.scan_parquet(), Polars' query optimizer applied Predicate Pushdown and Column Pruning—it only read the strictly required columns from the disk and skipped entire row groups using Parquet metadata. However, because the queries ended with .collect(), the final result set was still fully materialized in Python's memory.

In [5]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
For queries with massive outputs (like Task 3.1 returning 16.6 million rows), collect(engine="streaming") did not reduce peak memory; in fact, it increased it. While the streaming engine processes intermediate steps in memory-efficient batches, calling .collect() forces the engine to eventually concatenate all those batches into a single contiguous DataFrame in RAM. This causes a significant temporary memory spike and slight runtime overhead compared to standard lazy execution."""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

For queries with massive outputs (like Task 3.1 returning 16.6 million rows), collect(engine="streaming") did not reduce peak memory; in fact, it increased it. While the streaming engine processes intermediate steps in memory-efficient batches, calling .collect() forces the engine to eventually concatenate all those batches into a single contiguous DataFrame in RAM. This causes a significant temporary memory spike and slight runtime overhead compared to standard lazy execution.

In [4]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
A streaming sink (sink_parquet) is absolutely required when the final result set is larger than the available physical RAM . In our tests, sink_parquet kept the memory footprint flat  while writing a massive output file. It is the appropriate choice for ETL pipelines where the goal is to transform and store data for downstream systems, rather than materializing millions of rows in Python memory for interactive analysis."""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

A streaming sink (sink_parquet) is absolutely required when the final result set is larger than the available physical RAM . In our tests, sink_parquet kept the memory footprint flat  while writing a massive output file. It is the appropriate choice for ETL pipelines where the goal is to transform and store data for downstream systems, rather than materializing millions of rows in Python memory for interactive analysis.

In [10]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
Yes, local PySpark behaved exactly as expected: it was significantly slower than the single-node engines. For example, Q1 took ~0.07s in Polars (Lazy) and ~0.18s in DuckDB, but ~0.93s in local PySpark. This perfectly illustrates Spark's intrinsic overhead. Even on a single machine, Spark pays the penalty of Java Virtual Machine initialization, Directed Acyclic Graph generation, and task serialization. For a relatively "small" 1 GB dataset that fits entirely in RAM, this distributed scheduling overhead vastly outweighs the benefits of parallelization.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

Yes, local PySpark behaved exactly as expected: it was significantly slower than the single-node engines. For example, Q1 took ~0.07s in Polars (Lazy) and ~0.18s in DuckDB, but ~0.93s in local PySpark. This perfectly illustrates Spark's intrinsic overhead. Even on a single machine, Spark pays the penalty of Java Virtual Machine initialization, Directed Acyclic Graph generation, and task serialization. For a relatively "small" 1 GB dataset that fits entirely in RAM, this distributed scheduling overhead vastly outweighs the benefits of parallelization.

In [2]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
We would recommend migrating to a distributed cluster when the dataset hits the Storage or Materialization Bottleneck typically around the 50-100+ GB mark, depending on node hardware. Specifically, you should switch to a cluster when: 1) intermediate shuffle data or final materialized results exceed the physical RAM of a single node 2) data must be read from distributed cloud storage where cluster network bandwidth outpaces a single machine, or 3) the job requires fault tolerance to survive potential node crashes during long-running ETL tasks."""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

We would recommend migrating to a distributed cluster when the dataset hits the Storage or Materialization Bottleneck typically around the 50-100+ GB mark, depending on node hardware. Specifically, you should switch to a cluster when: 1) intermediate shuffle data or final materialized results exceed the physical RAM of a single node 2) data must be read from distributed cloud storage where cluster network bandwidth outpaces a single machine, or 3) the job requires fault tolerance to survive potential node crashes during long-running ETL tasks.

In [9]:
from IPython.display import Markdown, display


def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))


# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
The PyArrow backend improved runtime across all queries, but exposed a fascinating CPU vs. Memory trade-off depending on the operation. For filtering (Q1) and array explosion (Q3), PyArrow overwhelmingly outperformed the default NumPy/object backend in both speed and memory (e.g., Q1 memory dropped from ~1919 MB to ~172 MB).
However, in the string-heavy Join operation (Q2), the PyArrow backend's peak memory actually increased massively (~930 MB compared to the default's ~176 MB), even though execution time dropped from ~32.8s to ~13.8s. This happens because the default Pandas object dtype stores strings as simple memory pointers (references), so merging them takes very little new RAM but is incredibly slow due to Python execution overhead. PyArrow, conversely, allocates contiguous native memory blocks for the joined string data. This creates a larger memory usage"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

The PyArrow backend improved runtime across all queries, but exposed a fascinating CPU vs. Memory trade-off depending on the operation. For filtering (Q1) and array explosion (Q3), PyArrow overwhelmingly outperformed the default NumPy/object backend in both speed and memory (e.g., Q1 memory dropped from ~1919 MB to ~172 MB).

However, in the string-heavy Join operation (Q2), the PyArrow backend's peak memory actually increased massively (~930 MB compared to the default's ~176 MB), even though execution time dropped from ~32.8s to ~13.8s. This happens because the default Pandas object dtype stores strings as simple memory pointers (references), so merging them takes very little new RAM but is incredibly slow due to Python execution overhead. PyArrow, conversely, allocates contiguous native memory blocks for the joined string data. This creates a larger memory usage